# **Import & Parametri**

In [6]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.sql.types import StringType

spark = SparkSession.builder.getOrCreate()

# ---------------------------------------------------------------------------
# Parametri principali — sovrascrivibili da una Pipeline
# ---------------------------------------------------------------------------
BRONZE_LAKEHOUSE = "LH_Bronze"
SILVER_LAKEHOUSE = "LH_Silver"
SOURCE_FOLDER    = "sales"
CONFIG_TABLE     = "pipeline_config"

config_path = "Files/config_ingestion.csv"

# ---------------------------------------------------------------------------
# Parametri ggiuntivi per tabella audit
# ---------------------------------------------------------------------------

from datetime import datetime
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, FloatType, TimestampType
)
import traceback

# Nuovo parametro: nome della tabella di audit (stessa del notebook 1)
AUDIT_TABLE = "silver_audit_log"


StatementMeta(, 416fc05c-d4ea-4f5d-97ed-f171d18b5402, 8, Finished, Available, Finished, False)

# **Path ABFSS dei Lakehouse**

In [7]:
bronze_info = notebookutils.lakehouse.get(BRONZE_LAKEHOUSE)
bronze_path = bronze_info["properties"]["abfsPath"]

silver_info = notebookutils.lakehouse.get(SILVER_LAKEHOUSE)
silver_path = silver_info["properties"]["abfsPath"]

print(f"📂 Bronze path : {bronze_path}")
print(f"📂 Silver path : {silver_path}")


StatementMeta(, 416fc05c-d4ea-4f5d-97ed-f171d18b5402, 9, Finished, Available, Finished, False)

📂 Bronze path : abfss://47f4b971-7b6b-4af9-8001-77745ea917c0@onelake.dfs.fabric.microsoft.com/c9c219f7-14cc-47f2-8994-35d08101e937
📂 Silver path : abfss://47f4b971-7b6b-4af9-8001-77745ea917c0@onelake.dfs.fabric.microsoft.com/b4c95460-8615-46fa-9884-21f085344ad0


# **Lettura Configurazione & Selezione Tabelle sales**

In [8]:
df_config = (
    spark.read
    .option("header",      "true")
    .option("inferSchema", "true")
    .option("sep",         ";")
    .csv(config_path)
)

rows = (
    df_config
    .filter(df_config.SourceFolder == SOURCE_FOLDER)
    .collect()
)

object_names = [row["DestinationTable"] for row in rows]
print(f"📋 Tabelle da processare: {len(object_names)} → {object_names}")


StatementMeta(, 416fc05c-d4ea-4f5d-97ed-f171d18b5402, 10, Finished, Available, Finished, False)

📋 Tabelle da processare: 4 → ['bronze_orders', 'bronze_order_details', 'bronze_reviews', 'bronze_order_campaign']


# **Funzione di pulizia: approccio funzionale a singola select**

In [9]:

def pulisci_dataframe(df: DataFrame) -> DataFrame:
    """
    Pulizia superficiale con approccio funzionale a passaggio unico:
      - Per le colonne stringa: TRIM + normalizzazione stringa vuota → NULL,
        costruiti come UNA SOLA espressione per colonna (trim + when annidati),
        applicata in un'unica chiamata a .select() su tutto il DataFrame.
      - Le colonne non-stringa passano invariate (F.col(nome)).
    Vantaggio rispetto al ciclo di withColumn: un solo piano logico Spark,
    niente colonne "ricreate" più volte in sequenza → meno overhead sul
    query optimizer (Catalyst) e codice più leggibile su tabelle larghe.
    """
    espressioni = []

    for campo in df.schema.fields:
        nome = campo.name

        if isinstance(campo.dataType, StringType):
            # Trim + empty-string→NULL in un'unica espressione concatenata
            col_pulita = F.trim(F.col(nome))
            col_pulita = F.when(col_pulita == "", None).otherwise(col_pulita)
            espressioni.append(col_pulita.alias(nome))
        else:
            # Colonne non testuali: nessuna modifica necessaria
            espressioni.append(F.col(nome))

    df_pulito = df.select(*espressioni)

    # Righe fantasma: eliminate in un secondo momento, dopo la normalizzazione,
    # così anche le righe con soli spazi bianchi (" ", "") vengono correttamente
    # riconosciute come NULL prima del controllo "tutta la riga è vuota"
    df_pulito = df_pulito.dropna(how="all")

    return df_pulito


StatementMeta(, 416fc05c-d4ea-4f5d-97ed-f171d18b5402, 11, Finished, Available, Finished, False)

# **Processing Bronze → Silver (Full Load**

In [10]:

print("=" * 65)
print("🚀 INIZIO PROCESSING BRONZE → SILVER")
print("=" * 65)

risultati = []   # ➕ NUOVO: raccoglie l'esito di ogni tabella per l'audit log

for row in rows:
    nome_tabella = row["DestinationTable"]
    nome_silver  = nome_tabella.replace("bronze_", "silver_")
    inizio       = datetime.now()   # ➕ NUOVO

    print(f"\n⏳ Processing: {nome_tabella} → {nome_silver}")

    try:
        # 1. Lettura dal Bronze via path ABFSS
        df_bronze = spark.read.format("delta").load(f"{bronze_path}/Tables/{nome_tabella}")
        righe_bronze = df_bronze.count()

        # 2. Pulizia superficiale (approccio a singola select)
        df_silver = pulisci_dataframe(df_bronze)
        righe_silver = df_silver.count()

        # 3. Scrittura Full nel Silver
        (
            df_silver.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .save(f"{silver_path}/Tables/{nome_silver}")
        )

        fine     = datetime.now()                                  # ➕ NUOVO
        durata_s = round((fine - inizio).total_seconds(), 2)       # ➕ NUOVO

        print(f"   ✅ Completato in {durata_s}s")
        print(f"   ├── Righe Bronze : {righe_bronze:,}")
        print(f"   ├── Righe Silver : {righe_silver:,}")
        print(f"   └── Scartate     : {righe_bronze - righe_silver:,}")

        risultati.append({                                          # ➕ NUOVO
            "tabella"       : nome_silver,
            "righe_bronze"  : righe_bronze,
            "righe_silver"  : righe_silver,
            "righe_scartate": righe_bronze - righe_silver,
            "durata_sec"    : durata_s,
            "stato"         : "SUCCESS",
            "errore"        : "",       # stringa vuota, non None: mantiene lo schema StringType
            "processed_ts"  : fine
        })

    except Exception as e:
        fine = datetime.now()           # ➕ NUOVO
        msg  = traceback.format_exc()   # ➕ NUOVO
        print(f"   ❌ ERRORE su {nome_tabella}: {e}")

        risultati.append({               # ➕ NUOVO
            "tabella"       : nome_silver,
            "righe_bronze"  : 0,
            "righe_silver"  : 0,
            "righe_scartate": 0,
            "durata_sec"    : round((fine - inizio).total_seconds(), 2),
            "stato"         : "FAILED",
            "errore"        : msg,
            "processed_ts"  : fine
        })
        continue

print("\n" + "=" * 65)
print("🏁 PROCESSING COMPLETATO")
print("=" * 65)


StatementMeta(, 416fc05c-d4ea-4f5d-97ed-f171d18b5402, 12, Finished, Available, Finished, False)

🚀 INIZIO PROCESSING BRONZE → SILVER

⏳ Processing: bronze_orders → silver_orders
   ✅ Completato in 5.11s
   ├── Righe Bronze : 100,000
   ├── Righe Silver : 100,000
   └── Scartate     : 0

⏳ Processing: bronze_order_details → silver_order_details
   ✅ Completato in 5.12s
   ├── Righe Bronze : 249,521
   ├── Righe Silver : 249,521
   └── Scartate     : 0

⏳ Processing: bronze_reviews → silver_reviews
   ✅ Completato in 4.09s
   ├── Righe Bronze : 69,636
   ├── Righe Silver : 69,636
   └── Scartate     : 0

⏳ Processing: bronze_order_campaign → silver_order_campaign
   ✅ Completato in 4.36s
   ├── Righe Bronze : 72,034
   ├── Righe Silver : 72,034
   └── Scartate     : 0

🏁 PROCESSING COMPLETATO


# **Salvataggio Audit Log**

In [11]:

# ---------------------------------------------------------------------------
# Stesso schema del notebook 1: garantisce compatibilità di append
# sulla tabella "silver_audit_log" condivisa tra i due notebook.
# ---------------------------------------------------------------------------
schema_audit = StructType([
    StructField("tabella",        StringType(),    True),
    StructField("righe_bronze",   IntegerType(),   True),
    StructField("righe_silver",   IntegerType(),   True),
    StructField("righe_scartate", IntegerType(),   True),
    StructField("durata_sec",     FloatType(),     True),
    StructField("stato",          StringType(),    True),
    StructField("errore",         StringType(),    True),
    StructField("processed_ts",   TimestampType(), True),
])

successi = [r for r in risultati if r["stato"] == "SUCCESS"]
falliti  = [r for r in risultati if r["stato"] == "FAILED"]

print(f"\n📊 RIEPILOGO ESECUZIONE")
print(f"   ✅ Tabelle OK     : {len(successi)}/{len(risultati)}")
print(f"   ❌ Tabelle Fallite: {len(falliti)}/{len(risultati)}")

if falliti:
    print(f"\n⚠  TABELLE CON ERRORI:")
    for r in falliti:
        print(f"   → {r['tabella']}: {r['errore'][:120]}...")

df_audit = spark.createDataFrame(risultati, schema=schema_audit)

(
    df_audit.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .save(f"{silver_path}/Tables/{AUDIT_TABLE}")
)

print(f"\n📝 Audit log salvato in: Silver → {AUDIT_TABLE}")

# Solleva eccezione se ci sono fallimenti, per permettere alla Pipeline
# Data Factory di rilevare l'errore e attivare eventuali notifiche/retry.
if falliti:
    raise Exception(
        f"❌ {len(falliti)} tabelle non processate. Consulta '{AUDIT_TABLE}' per i dettagli."
    )


StatementMeta(, 416fc05c-d4ea-4f5d-97ed-f171d18b5402, 13, Finished, Available, Finished, False)


📊 RIEPILOGO ESECUZIONE
   ✅ Tabelle OK     : 4/4
   ❌ Tabelle Fallite: 0/4

📝 Audit log salvato in: Silver → silver_audit_log
